In [0]:
%pip install hdbcli

In [0]:
from hdbcli import dbapi
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

TABELAS = ["KNA1", "KNB1", "VBAK", "VBAP", "VBRK", "VBRP", "BSID", "BSAD"]
SCHEMA_HANA = "SAPHANADB"
SISTEMA = "s4lake-hana"

conn = dbapi.connect(
    address=dbutils.secrets.get("s4lake", "hana_host"),
    port=443,
    user="S4LAKE_LEITURA",
    password=dbutils.secrets.get("s4lake", "hana_senha_leitura"),
    encrypt=True,
    sslValidateCertificate=True,
)
cur = conn.cursor()

for tabela in TABELAS:
    cur.execute(f'SELECT * FROM "{SCHEMA_HANA}"."{tabela}"')
    colunas = [d[0] for d in cur.description]
    linhas = [
        tuple(None if v is None or v == "" else str(v) for v in registro)
        for registro in cur.fetchall()
    ]

    schema = StructType([StructField(c, StringType(), True) for c in colunas])
    df = (
        spark.createDataFrame(linhas, schema)
        .withColumn("_ingestao_em", F.current_timestamp())
        .withColumn("_sistema_origem", F.lit(SISTEMA))
        .withColumn("_tabela_origem", F.lit(f"{SCHEMA_HANA}.{tabela}"))
    )
    (
        df.write.mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"workspace.s4lake_bronze.{tabela.lower()}")
    )
    print(f"{tabela}: {df.count():,} linhas")

conn.close()